<a href="https://colab.research.google.com/github/takuya0724/sticker-pro/blob/main/StickerPro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

print("キャラクター画像を選択してください")

uploaded = files.upload()

filename = list(uploaded.keys())[0]

img = Image.open(filename)

print(f"読み込み完了：{filename}")
print(f"画像サイズ：{img.size}")

plt.imshow(img)
plt.axis("off")
plt.show()
from PIL import Image
from google.colab import files
import os
import zipfile

LINE_SIZE = 512
PADDING = 40   # 余白（小さくするとキャラが大きくなる）

print("16分割画像を選択してください")

uploaded = files.upload()
filename = list(uploaded.keys())[0]

img = Image.open(filename).convert("RGBA")

w, h = img.size

cols = 4
rows = 4

cell_w = w // cols
cell_h = h // rows

os.makedirs("stickers", exist_ok=True)

count = 1

for y in range(rows):
    for x in range(cols):

        crop = img.crop((
            x * cell_w,
            y * cell_h,
            (x + 1) * cell_w,
            (y + 1) * cell_h
        ))

        # 透明部分を除いた範囲を取得
        bbox = crop.getbbox()

        if bbox:
            crop = crop.crop(bbox)

        # LINE用キャンバス
        canvas = Image.new("RGBA", (LINE_SIZE, LINE_SIZE), (0,0,0,0))

        cw, ch = crop.size

        scale = min(
            (LINE_SIZE - PADDING*2) / cw,
            (LINE_SIZE - PADDING*2) / ch
        )

        nw = int(cw * scale)
        nh = int(ch * scale)

        crop = crop.resize((nw, nh), Image.LANCZOS)

        px = (LINE_SIZE - nw) // 2
        py = (LINE_SIZE - nh) // 2

        canvas.paste(crop, (px, py), crop)

        canvas.save(f"stickers/sticker_{count:02}.png")

        count += 1

zipname = "stickers.zip"

with zipfile.ZipFile(zipname, "w") as z:
    for f in sorted(os.listdir("stickers")):
        z.write(os.path.join("stickers", f), f)

files.download(zipname)

print("完成！")

キャラクター画像を選択してください


In [ ]:
from PIL import Image

# LINEスタンプサイズへ変換
sticker = img.copy()
sticker.thumbnail((512, 512), Image.LANCZOS)

# 透明背景対応
canvas = Image.new("RGBA", (512, 512), (255, 255, 255, 0))

# 中央に配置
x = (512 - sticker.width) // 2
y = (512 - sticker.height) // 2
canvas.paste(sticker, (x, y))

# 保存
canvas.save("sticker_512.png")

print("✅ LINEスタンプサイズに変換しました！")

plt.figure(figsize=(5,5))
plt.imshow(canvas)
plt.axis("off")
plt.show()

In [ ]:
# LINEスタンプ用セリフ16種類

messages = [
    "おはよう！",
    "こんにちは！",
    "こんばんは！",
    "ありがとう！",
    "了解！",
    "おつかれさま！",
    "ごめんね！",
    "よろしく！",
    "OK！",
    "がんばろう！",
    "最高！",
    "うれしい！",
    "またね！",
    "おやすみ！",
    "びっくり！",
    "酔ってきたー！"
]

print("===== 作成するスタンプ =====")

for i, message in enumerate(messages, 1):
    print(f"{i}. {message}")

In [ ]:
master_prompt = """
Keep exactly the same monkey character.
Maintain the same face, fur color, proportions and watercolor style.
Transparent background.
High quality LINE sticker illustration.
"""

print("===== AI画像生成プロンプト =====\n")

for i, message in enumerate(messages, 1):
    prompt = f"""
{master_prompt}

Expression:
{message}

No text in the image.
"""
    print(f"------ {i} ------")
    print(prompt)

In [ ]:
import csv

rows = []

for i, message in enumerate(messages, 1):

    prompt = f"""
Keep exactly the same monkey character.
Maintain the same face, fur color, proportions and watercolor style.
Transparent background.
High quality LINE sticker illustration.

Expression:
{message}

No text in the image.
"""

    rows.append([i, message, prompt])

with open("StickerPro_prompts.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["No", "Message", "Prompt"])
    writer.writerows(rows)

print("✅ StickerPro_prompts.csv を作成しました！")

In [ ]:
from google.colab import files

files.download("StickerPro_prompts.csv")

In [ ]:
# ===== キャラクター設定 =====

character = {
    "name": "サル",
    "style": "Soft watercolor illustration",
    "background": "Transparent",
    "text": "No text",
    "face": "Round face",
    "fur": "Brown fur with light beige face",
    "eyes": "Large black eyes",
    "ears": "Small rounded ears",
    "nose": "Small brown nose",
    "consistency": "Keep exactly the same character in every image."
}

print("✅ キャラクター設定を読み込みました！")
print(character)

In [ ]:
print("===== StickerPro Prompt Generator =====\n")

for i, message in enumerate(messages, 1):

    prompt = f"""
Character:
{character['name']}

Style:
{character['style']}

Appearance:
{character['face']}
{character['fur']}
{character['eyes']}
{character['ears']}
{character['nose']}

{character['consistency']}

Background:
{character['background']}

Expression:
{message}

{character['text']}
"""

    print(f"========== {i} ==========")
    print(prompt)